# Ultimate NAKALA Workflow

Complete workflow for uploading, organizing, and curating data in NAKALA repository.
This notebook demonstrates all steps from initial upload to quality analysis.

In [1]:
# Setup and imports
import sys
import os
from pathlib import Path
import logging

# Add workflow modules to path
sys.path.append('workflow_modules')

# Import workflow modules
from workflow_config import WorkflowConfig
from data_uploader import DataUploader
from collection_manager import CollectionManager
from curator_operations import CuratorOperations
from metadata_enhancer import MetadataEnhancer
from quality_analyzer import QualityAnalyzer
from workflow_summary import WorkflowSummary

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("🚀 Ultimate NAKALA Workflow - All modules loaded successfully!")

🚀 Ultimate NAKALA Workflow - All modules loaded successfully!


## Step 1: Configuration Setup

Configure the workflow with API credentials and dataset paths.

In [2]:
# Initialize workflow configuration
api_key = os.getenv('NAKALA_API_KEY', '33170cfe-f53c-550b-5fb6-4814ce981293')
base_path = '../sample_dataset'

# Create workflow configuration
workflow_config = WorkflowConfig(api_key=api_key, base_path=base_path)
config = workflow_config.get_config_dict()

# Add additional configuration parameters
config.update({
    'mode': 'folder',
    'batch_size': 10,
    'max_retries': 3,
    'timeout': 300
})

print("📋 Workflow Configuration:")
print(f"   API URL: {config['api_url']}")
print(f"   Base Path: {config['base_path']}")
print(f"   Dataset CSV: {config['dataset_csv']}")
print(f"   Collections CSV: {config['collections_csv']}")
print(f"   Mode: {config['mode']}")
print(f"   Batch Size: {config['batch_size']}")

2025-06-25 11:21:41,818 - workflow_config - INFO - ✅ Configuration validated successfully


📋 Workflow Configuration:
   API URL: https://apitest.nakala.fr
   Base Path: ../sample_dataset
   Dataset CSV: ../sample_dataset/folder_data_items.csv
   Collections CSV: ../sample_dataset/folder_collections.csv
   Mode: folder
   Batch Size: 10


## Step 2: Data Upload

Upload datasets to NAKALA repository using the configured parameters.

In [3]:
# Initialize data uploader
uploader = DataUploader(config)

print("📤 Starting data upload process...")

# Perform upload operation
try:
    upload_results = uploader.upload_datasets(
        mode=config['mode']  # Use folder mode from configuration
    )
    
    print(f"✅ Upload completed successfully!")
    print(f"   Items uploaded: {upload_results['stats']['successful_uploads']}")
    print(f"   Failed uploads: {upload_results['stats']['failed_uploads']}")
    print(f"   Results file: {upload_results['stats']['results_file']}")
    
except Exception as e:
    print(f"❌ Upload failed: {e}")
    upload_results = None

2025-06-25 11:21:41,822 - data_uploader - INFO - 📤 Starting dataset upload...
2025-06-25 11:21:41,822 - data_uploader - INFO - Executing: o-nakala-upload --api-key 33170cfe-f53c-550b-5fb6-4814ce981293 --dataset ../sample_dataset/folder_data_items.csv --mode folder --base-path ../sample_dataset --output ../sample_dataset/upload_results.csv --folder-config ../sample_dataset/folder_data_items.csv


📤 Starting data upload process...


2025-06-25 11:21:43,079 - data_uploader - INFO - ✅ Upload completed successfully



📊 Upload Summary
Total Datasets: 5
Successful: 0
Failed: 0
Execution Time: 1.26 seconds
First Dataset ID: 10.34847/nkl.fb48hqx3

📋 First Few Results:
           identifier                                                                                                                                                                                                            files                                    title status                                                                             response
10.34847/nkl.fb48hqx3                                                                                                                                                   site_photograph_1.jpg,a08a025dac873238a820d4feea69dc1a33882029         fr:Images de test|en:Test Images     OK {"code": 201, "message": "Data created", "payload": {"id": "10.34847/nkl.fb48hqx3"}}
10.34847/nkl.b5f9bd0p                                                                                    preprocess_data.

## Step 3: Collection Management

Create and organize collections based on uploaded data.

In [4]:
# Initialize collection manager
collection_manager = CollectionManager(config)

print("📁 Starting collection management...")

# Create collections from upload results
if upload_results and upload_results['success']:
    try:
        collection_results = collection_manager.create_collections(
            upload_results_file=upload_results['stats']['results_file']
        )
        
        print(f"✅ Collection creation completed!")
        print(f"   Collections created: {collection_results['stats']['successful_collections']}")
        print(f"   Failed collections: {collection_results['stats']['failed_collections']}")
        
    except Exception as e:
        print(f"❌ Collection creation failed: {e}")
        collection_results = None
else:
    print("⏭️  Skipping collection creation - no upload results available")
    collection_results = None

2025-06-25 11:21:43,089 - collection_manager - INFO - 📁 Starting collection creation...
2025-06-25 11:21:43,089 - collection_manager - INFO - Executing: o-nakala-collection --api-key 33170cfe-f53c-550b-5fb6-4814ce981293 --from-upload-output ../sample_dataset/upload_results.csv --from-folder-collections ../sample_dataset/folder_collections.csv --collection-report ../sample_dataset/collections_output.csv


📁 Starting collection management...


2025-06-25 11:21:43,508 - collection_manager - INFO - ✅ Collection creation completed successfully



📁 Collection Creation Summary
Total Collections: 3
Successful: 0
Failed: 0
Execution Time: 0.42 seconds
First Collection ID: 10.34847/nkl.6fa5c7ay

📋 First Few Collections:
        collection_id                                          collection_title  status  data_items_count                              data_items_ids creation_status  error_message           timestamp
10.34847/nkl.6fa5c7ay fr:Collection Code et Données|en:Code and Data Collection private                 2 10.34847/nkl.b5f9bd0p;10.34847/nkl.0026zah1         SUCCESS            NaN 2025-06-25 11:21:43
10.34847/nkl.adcftbpw           fr:Collection Documents|en:Documents Collection private                 1                       10.34847/nkl.bb107ijm         SUCCESS            NaN 2025-06-25 11:21:43
10.34847/nkl.ea8diq4d         fr:Collection Multimédia|en:Multimedia Collection private                 2 10.34847/nkl.fb48hqx3;10.34847/nkl.9821o4no         SUCCESS            NaN 2025-06-25 11:21:43
✅ Collection creation 

## Step 4: Metadata Enhancement

Enhance and standardize metadata for uploaded items.

In [5]:
# Initialize metadata enhancer
metadata_enhancer = MetadataEnhancer(config)

print("🔍 Starting metadata enhancement...")

# Enhance metadata for uploaded items
try:
    # Generate data enhancements
    data_enhancement_results = metadata_enhancer.generate_data_enhancements(
        upload_results_file=upload_results['stats']['results_file'] if upload_results else None
    )
    
    # Generate collection enhancements if collections were created
    collection_enhancement_results = None
    if collection_results and collection_results.get('success'):
        collection_enhancement_results = metadata_enhancer.generate_collection_enhancements()
    
    enhancement_results = {
        'data_enhancements': data_enhancement_results,
        'collection_enhancements': collection_enhancement_results,
        'success': True
    }
    
    print(f"✅ Metadata enhancement completed!")
    print(f"   Data enhancements: {data_enhancement_results.get('stats', {}).get('enhancements_generated', 0)}")
    if collection_enhancement_results:
        print(f"   Collection enhancements: {collection_enhancement_results.get('stats', {}).get('enhancements_generated', 0)}")
    
except Exception as e:
    print(f"❌ Metadata enhancement failed: {e}")
    enhancement_results = None

2025-06-25 11:21:43,518 - metadata_enhancer - INFO - 🤖 Generating dataset metadata enhancements...
2025-06-25 11:21:43,537 - metadata_enhancer - INFO - ✅ Dataset enhancements generated successfully
2025-06-25 11:21:43,539 - metadata_enhancer - INFO - 🤖 Generating collection metadata enhancements...
2025-06-25 11:21:43,557 - metadata_enhancer - INFO - ✅ Collection enhancements generated successfully


🔍 Starting metadata enhancement...
✅ Metadata enhancement completed!
   Data enhancements: 0
   Collection enhancements: 0


## Step 5: Curator Operations

Perform advanced curation operations including validation and duplicate detection.

In [6]:
# Initialize curator operations
curator = CuratorOperations(config)

print("🔧 Starting curator operations...")

# Perform comprehensive curation
try:
    # Apply all curation operations (datasets and collections)
    curation_results = curator.curate_all()
    
    print(f"✅ Curator operations completed!")
    
    # Display dataset curation results
    if curation_results['dataset_curation']:
        dataset_stats = curation_results['dataset_curation']['stats']
        print(f"   Dataset modifications: {dataset_stats.get('modifications_applied', 0)}")
    
    # Display collection curation results
    if curation_results['collection_curation']:
        collection_stats = curation_results['collection_curation']['stats']
        print(f"   Collection modifications: {collection_stats.get('modifications_applied', 0)}")
    
    validation_results = curation_results
    duplicate_results = None  # Not available in this workflow
    
except Exception as e:
    print(f"❌ Curator operations failed: {e}")
    validation_results = None
    duplicate_results = None

2025-06-25 11:21:43,563 - curator_operations - INFO - 🚀 Starting comprehensive metadata curation...
2025-06-25 11:21:43,563 - curator_operations - INFO - ✨ Starting dataset metadata curation...
2025-06-25 11:21:43,564 - curator_operations - INFO - Executing: o-nakala-curator --api-key 33170cfe-f53c-550b-5fb6-4814ce981293 --batch-modify ../sample_dataset/auto_data_modifications.csv --scope datasets


🔧 Starting curator operations...


2025-06-25 11:21:45,212 - curator_operations - INFO - ✅ Datasets curation completed successfully
2025-06-25 11:21:45,215 - curator_operations - INFO - 📁 Starting collection metadata curation...
2025-06-25 11:21:45,216 - curator_operations - INFO - Executing: o-nakala-curator --api-key 33170cfe-f53c-550b-5fb6-4814ce981293 --batch-modify ../sample_dataset/auto_collection_modifications.csv --scope collections
2025-06-25 11:21:46,608 - curator_operations - INFO - ✅ Collections curation completed successfully



✨ Metadata Curation Summary
Dataset Curation:
  - Modifications Applied: 5
  - Datasets Modified: 0
  - Execution Time: 1.65s
Collection Curation:
  - Modifications Applied: 3
  - Collections Modified: 0
  - Execution Time: 1.39s
✅ Curator operations completed!
   Dataset modifications: 0
   Collection modifications: 0


## Step 6: Quality Analysis

Generate comprehensive quality reports and analysis.

In [7]:
# Initialize quality analyzer
quality_analyzer = QualityAnalyzer(config)

print("📊 Starting quality analysis...")

# Generate quality report
try:
    quality_results = quality_analyzer.generate_quality_report(
        scope='all',  # Analyze all items and collections
        output_file=None  # Use default output file
    )
    
    print(f"✅ Quality analysis completed!")
    
    # Analyze quality trends
    trends = quality_analyzer.analyze_quality_trends()
    if trends and isinstance(trends, dict):
        print(f"\n📈 Quality Trends:")
        print(f"   Metadata Completeness: {trends.get('metadata_completeness', 'Unknown')}")
        print(f"   Common Issues: {len(trends.get('common_issues', []))}")
        print(f"   Improvement Areas: {len(trends.get('improvement_areas', []))}")
    
    # Export quality summary
    summary_file = quality_analyzer.export_quality_summary()
    print(f"   Quality summary exported to: {summary_file}")
    
except Exception as e:
    print(f"❌ Quality analysis failed: {e}")
    quality_results = None

2025-06-25 11:21:46,616 - quality_analyzer - INFO - 📊 Generating quality analysis report...
2025-06-25 11:21:46,617 - quality_analyzer - INFO - Executing: o-nakala-curator --api-key 33170cfe-f53c-550b-5fb6-4814ce981293 --quality-report --scope all --output ../sample_dataset/quality_report.json


📊 Starting quality analysis...


2025-06-25 11:21:47,434 - quality_analyzer - INFO - ✅ Quality analysis completed successfully
2025-06-25 11:21:47,435 - quality_analyzer - ERROR - Error analyzing quality trends: 'str' object has no attribute 'get'
2025-06-25 11:21:47,436 - quality_analyzer - WARNING - No item-level data available for export



📊 Quality Analysis Summary
Scope: ALL
Items Analyzed: 98
Overall Quality Score: 0.00
Issues Found: 98
Recommendations: 4
Analysis Time: 0.82 seconds

💡 Top Recommendations:
  1. Fix metadata errors in 33 collections
  2. Fix metadata errors in 65 datasets
  3. Consider improving metadata quality - current score is below 80%
✅ Quality analysis completed!

📈 Quality Trends:
   Metadata Completeness: Needs Improvement
   Common Issues: 0
   Improvement Areas: 0
   Quality summary exported to: ../sample_dataset/quality_summary.csv


## Step 7: Workflow Summary

Generate comprehensive workflow summary and final report.

In [8]:
# Initialize workflow summary
workflow_summary = WorkflowSummary(config)

print("📋 Generating workflow summary...")

# Generate comprehensive summary
try:
    summary_results = workflow_summary.generate_comprehensive_summary()
    
    print(f"✅ Workflow summary generated!")
    
    # Display final statistics
    print(f"\n🎯 Final Workflow Statistics:")
    overall_stats = summary_results.get('overall_statistics', {})
    print(f"   Total Operations: {overall_stats.get('total_operations', 'Unknown')}")
    print(f"   Successful Operations: {overall_stats.get('successful_operations', 'Unknown')}")
    print(f"   Items Created: {overall_stats.get('total_items_created', 'Unknown')}")
    print(f"   Enhancements Applied: {overall_stats.get('total_enhancements_applied', 'Unknown')}")
    print(f"   Success Rate: {overall_stats.get('workflow_success_rate', 0):.1f}%")
    
    # Display operation summaries
    if summary_results.get('upload_summary', {}).get('available'):
        upload_stats = summary_results['upload_summary']
        print(f"   Upload: {upload_stats.get('total_datasets', 0)} datasets")
    
    if summary_results.get('collections_summary', {}).get('available'):
        collection_stats = summary_results['collections_summary']
        print(f"   Collections: {collection_stats.get('total_collections', 0)} collections")
    
    if summary_results.get('quality_summary', {}).get('available'):
        quality_stats = summary_results['quality_summary']
        print(f"   Quality Score: {quality_stats.get('overall_quality_score', 0):.1f}")
    
except Exception as e:
    print(f"❌ Workflow summary failed: {e}")
    summary_results = None

2025-06-25 11:21:47,441 - workflow_summary - INFO - 🎯 Generating comprehensive workflow summary...
2025-06-25 11:21:47,445 - workflow_summary - INFO - Complete summary saved to: ../sample_dataset/workflow_complete_summary.json


📋 Generating workflow summary...

🏆 ULTIMATE WORKFLOW COMPLETE SUMMARY
📊 Overall Success Rate: 71.4%
✅ Successful Operations: 5/7
📦 Total Items Created: 8
✨ Total Enhancements Applied: 8

📋 Operation Details:
----------------------------------------
📤 Data Upload: 5 datasets
   First Dataset: 10.34847/nkl.fb48hqx3
📁 Collections: 3 created
   First Collection: 10.34847/nkl.6fa5c7ay
📊 Quality Analysis: Score 0.00
   Issues Found: 0
✨ Dataset Enhancements: 5 applied
✨ Collection Enhancements: 3 applied
✅ Workflow summary generated!

🎯 Final Workflow Statistics:
   Total Operations: 7
   Successful Operations: 5
   Items Created: 8
   Enhancements Applied: 8
   Success Rate: 71.4%
   Upload: 5 datasets
   Collections: 3 collections
   Quality Score: 0.0


## Workflow Completion

The ultimate NAKALA workflow has been completed. All steps from upload to quality analysis have been executed.

In [9]:
print("\n🎉 Ultimate NAKALA Workflow Completed!")
print("=" * 60)

# Final verification
verification_steps = [
    ("Configuration", config is not None),
    ("Upload", upload_results is not None and upload_results.get('success', False)),
    ("Collections", collection_results is not None),
    ("Metadata Enhancement", enhancement_results is not None),
    ("Validation", validation_results is not None),
    ("Quality Analysis", quality_results is not None),
    ("Workflow Summary", summary_results is not None)
]

print("🔍 Workflow Step Verification:")
for step_name, success in verification_steps:
    status = "✅ COMPLETED" if success else "❌ FAILED"
    print(f"   {step_name}: {status}")

print("=" * 60)
print("📚 Check the following files for detailed results:")
print("   - upload_results.csv (upload details)")
print("   - collections_output.csv (collection details)")
print("   - quality_report.json (quality analysis)")
print("   - workflow_summary.json (comprehensive summary)")


🎉 Ultimate NAKALA Workflow Completed!
🔍 Workflow Step Verification:
   Configuration: ✅ COMPLETED
   Upload: ✅ COMPLETED
   Collections: ✅ COMPLETED
   Metadata Enhancement: ✅ COMPLETED
   Validation: ✅ COMPLETED
   Quality Analysis: ✅ COMPLETED
   Workflow Summary: ✅ COMPLETED
📚 Check the following files for detailed results:
   - upload_results.csv (upload details)
   - collections_output.csv (collection details)
   - quality_report.json (quality analysis)
   - workflow_summary.json (comprehensive summary)
